In [ ]:
# Install PyTorch 
!pip install torch torchvision torchaudio
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [2]:
import torch
import torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
cuda available: True
torch cuda: 12.8
device count: 1
gpu: NVIDIA GeForce RTX 5070 Laptop GPU


In [3]:
# Load CIFAR-10
import torch
import torchvision
import numpy as np

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

x_train = train_dataset.data
y_train = np.array(train_dataset.targets)

x_test = test_dataset.data
y_test = np.array(test_dataset.targets)

labels = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
num_classes = 10

c:\Users\wlsgm\anaconda3\envs\dlhw1\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
# Balanced sampling
SEED = 42
rng = np.random.default_rng(SEED)

TOTAL = 5000
PER_CLASS = TOTAL // num_classes

sel_list = []
for c in range(num_classes):
    idx = np.where(y_train == c)[0]
    sel_c = rng.choice(idx, size=PER_CLASS, replace=False)
    sel_list.append(sel_c)

sel = np.concatenate(sel_list)
rng.shuffle(sel)

x_small = x_train[sel]
y_small = y_train[sel]

print(x_small.shape)
print("per-class counts:", {c: int(np.sum(y_small == c)) for c in range(num_classes)})

(5000, 32, 32, 3)
per-class counts: {0: 500, 1: 500, 2: 500, 3: 500, 4: 500, 5: 500, 6: 500, 7: 500, 8: 500, 9: 500}


In [5]:
# Define preprocessing
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG = 224
BATCH = 32

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class CIFAR10Dataset(Dataset):
    def __init__(self, x, y, transform=None):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        label = self.y[idx]
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

train_ds = DataLoader(CIFAR10Dataset(x_small, y_small, transform=transform), 
                      batch_size=BATCH, shuffle=True)

test_ds  = DataLoader(CIFAR10Dataset(x_test, y_test, transform=transform), 
                      batch_size=BATCH, shuffle=False)

In [6]:
# Utility functions for performance evaluation

def per_class_accuracy(model, data_loader, y_true):
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for x, _ in data_loader:
            x = x.to(device)
            outputs = model(x)
            pred_idx = torch.argmax(outputs, dim=1)
            all_preds.extend(pred_idx.cpu().numpy())
    
    pred = np.array(all_preds)
    accs = {}
    for c in range(num_classes):
        idx = np.where(y_true == c)[0]
        accs[c] = float(np.mean(pred[idx] == c))
    return accs

def print_table(accs):
    for c in range(num_classes):
        print(f"{c:>2} {labels[c]:<11} {accs[c]*100:6.2f}%")


## 1. Model Training Configuration

### 1) Model Architecture
* **Backbone:** **ResNet50** (ImageNet-V2 weights) 

* **Two-Stage Training Strategy:**
    * **Stage 1 (Feature Extraction):** All backbone layers were **frozen**, and only the new classification head was trained. This allows the model to adapt to CIFAR-10 classes while preserving pre-trained knowledge.
    * **Stage 2 (Fine-tuning):** The **last 40 parameters** were **unfrozen** for fine-tuning. This stage optimizes the high-level features specifically for CIFAR-10 images to improve final accuracy.

### 2) Experimental Setup & Hyperparameters
* **Optimizer:** **Adam** optimizer was used for both stages.
* **Learning Rate (LR):**
    * **Stage 1:** $1 \times 10^{-3}$ (High LR for rapid head adaptation).
    * **Stage 2:** $2 \times 10^{-5}$ (Low LR for stable fine-tuning).
* **Epochs:** **10 Epochs** for Stage 1, followed by **15 Epochs** for Stage 2 (Total **25 Epochs**).
* **Loss Function:** **Cross-Entropy Loss**
* **Baseline Constraints:**
    * **No Augmentation:** No data transformation techniques (e.g., flipping, cropping) were applied except for resizing to 224x224.
    * **No Oversampling:** Used a balanced subset of 5,000 images, eliminating the need for class-balance adjustments.
    * **No Class Weights:** Assigned equal importance to all classes within the loss function.
    * **No Label Smoothing:** Used hard labels (One-hot encoding) for the baseline to establish a performance floor.

In [7]:
# Baseline ResNet50
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights

base = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
base.fc = nn.Linear(base.fc.in_features, num_classes)

# Freeze backbone
for param in base.parameters():
    param.requires_grad = False
for param in base.fc.parameters():
    param.requires_grad = True

baseline = base.to(device)

criterion = nn.CrossEntropyLoss()
print("Starting Stage 1: Training FC layer only...")

optimizer = optim.Adam(filter(lambda p: p.requires_grad, baseline.parameters()), lr=1e-3)

for epoch in range(10):
    baseline.train()
    total_loss = 0
    for x, y in train_ds:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = baseline(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Stage 1 - Epoch {epoch+1}/10, Loss: {total_loss/len(train_ds):.4f}")

# Unfreeze last 40 parameters
params = list(baseline.parameters())
for p in params[:-40]: p.requires_grad = False
for p in params[-40:]: p.requires_grad = True

# Stage 2: Fine-tuning 
print("\nStarting Stage 2: Fine-tuning last 40 layers...")
optimizer = optim.Adam(filter(lambda p: p.requires_grad, baseline.parameters()), lr=2e-5)

for epoch in range(15):
    baseline.train()
    total_loss = 0
    for x, y in train_ds:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = baseline(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Stage 2 - Epoch {epoch+1}/15, Loss: {total_loss/len(train_ds):.4f}")

Starting Stage 1: Training FC layer only...
Stage 1 - Epoch 1/10, Loss: 1.4015
Stage 1 - Epoch 2/10, Loss: 0.8855
Stage 1 - Epoch 3/10, Loss: 0.7545
Stage 1 - Epoch 4/10, Loss: 0.6743
Stage 1 - Epoch 5/10, Loss: 0.6235
Stage 1 - Epoch 6/10, Loss: 0.5800
Stage 1 - Epoch 7/10, Loss: 0.5446
Stage 1 - Epoch 8/10, Loss: 0.5085
Stage 1 - Epoch 9/10, Loss: 0.4814
Stage 1 - Epoch 10/10, Loss: 0.4751

Starting Stage 2: Fine-tuning last 40 layers...
Stage 2 - Epoch 1/15, Loss: 0.4071
Stage 2 - Epoch 2/15, Loss: 0.2648
Stage 2 - Epoch 3/15, Loss: 0.1913
Stage 2 - Epoch 4/15, Loss: 0.1444
Stage 2 - Epoch 5/15, Loss: 0.1023
Stage 2 - Epoch 6/15, Loss: 0.0768
Stage 2 - Epoch 7/15, Loss: 0.0706
Stage 2 - Epoch 8/15, Loss: 0.0543
Stage 2 - Epoch 9/15, Loss: 0.0430
Stage 2 - Epoch 10/15, Loss: 0.0337
Stage 2 - Epoch 11/15, Loss: 0.0269
Stage 2 - Epoch 12/15, Loss: 0.0262
Stage 2 - Epoch 13/15, Loss: 0.0236
Stage 2 - Epoch 14/15, Loss: 0.0194
Stage 2 - Epoch 15/15, Loss: 0.0164


In [8]:
accs_base = per_class_accuracy(baseline, test_ds, y_test)

print("=== Baseline per-class accuracy ===")
print_table(accs_base)

worst_label = min(accs_base, key=accs_base.get)
print("\nWorst label:", worst_label, labels[worst_label], f"acc = {accs_base[worst_label]:.4f}")

baseline_worst_acc = accs_base[worst_label]
print(f"Saved baseline {labels[worst_label]} acc = {baseline_worst_acc:.4f}")

=== Baseline per-class accuracy ===
 0 airplane     85.00%
 1 automobile   91.60%
 2 bird         79.60%
 3 cat          69.30%
 4 deer         79.80%
 5 dog          82.40%
 6 frog         85.20%
 7 horse        81.90%
 8 ship         89.50%
 9 truck        88.90%

Worst label: 3 cat acc = 0.6930
Saved baseline cat acc = 0.6930


## 2. Data-Centric Optimization

As the first model improvement, I performed data-centric optimization focusing on the `cat` class, which showed the lowest performance in the baseline, to evaluate the impact of data-level enhancements

### 1) Data Augmentation & Oversampling
* **Oversampling:** Duplicated the focus class samples to double the count (500 → 1,000).
* **Augmentation:** To prevent **overfitting** from duplication, I applied a pipeline including:
    * **Random Horizontal Flip & Rotation** (Spatial invariance)
    * **Random Resized Crop** (Scale robustness)
    * **Color Jitter** (Contrast adjustment)

### 2) Cost-Sensitive Learning (Class Weights)
* **Weighted Cross-Entropy:** Assigned a **3.0x weight** to the focus class.
* **Purpose:** Increases the penalty for misclassifying the focus class, forcing the model to prioritize its recall.

### 3) Training Strategy
Except for the data augmentation, the basic model architecture was kept identical to the BASE model.

In [9]:
# Select samples belonging to the focus (worst-performing) class
FOCUS = int(worst_label)

is_focus = (y_small == FOCUS)
x_focus = x_small[is_focus]
y_focus = y_small[is_focus]

# diff_1: Compared to the baseline, apply additional data augmentation
# Random horizontal flip, small rotation, zoom, and contrast adjustment
aug_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(18), 
    transforms.RandomResizedCrop((IMG, IMG), scale=(0.9, 1.0)), 
    transforms.ColorJitter(contrast=0.1), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# diff_2: Duplicate the focus class samples once
# This increases the total number of focus samples to 1,000
x_extra = x_focus.copy()
y_extra = y_focus.copy()

x_combined = np.concatenate([x_small, x_extra], axis=0)
y_combined = np.concatenate([y_small, y_extra], axis=0)

total_count = len(y_combined)
focus_count = np.sum(y_combined == FOCUS)

print(f"Total number of samples: {total_count}")
print(f"Number of FOCUS class samples: {focus_count}")

train_ds_focus_aug = DataLoader(
    CIFAR10Dataset(x_combined, y_combined, transform=aug_transform),
    batch_size=BATCH, 
    shuffle=True
)

Total number of samples: 5500
Number of FOCUS class samples: 1000


In [10]:
# diff_3: Adjust class weights to emphasize the focus class during training
weight_list = [1.0 for _ in range(num_classes)]
weight_list[FOCUS] = 3.0
class_weights = torch.tensor(weight_list, dtype=torch.float32).to(device)

# Same architecture as baseline
base2 = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
base2.fc = nn.Linear(base2.fc.in_features, num_classes)

for param in base2.parameters():
    param.requires_grad = False
for param in base2.fc.parameters():
    param.requires_grad = True

focused_aug_line = base2.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, focused_aug_line.parameters()), lr=1e-3)

print(f"Starting Focused Stage 1 (Focus Class: {labels[FOCUS]})...")
for epoch in range(10):
    focused_aug_line.train()
    total_loss = 0
    for x, y in train_ds_focus_aug:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = focused_aug_line(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Focused Stage 1 - Epoch {epoch+1}/10, Loss: {total_loss/len(train_ds_focus_aug):.4f}")

# Unfreeze last 40 parameters
params = list(focused_aug_line.parameters())
for p in params[:-40]: p.requires_grad = False
for p in params[-40:]: p.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, focused_aug_line.parameters()),
    lr=2e-5
)

print(f"\nStarting Focused Stage 2 (Fine-tuning)...")
for epoch in range(15):
    focused_aug_line.train()
    total_loss = 0
    for x, y in train_ds_focus_aug:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = focused_aug_line(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Focused Stage 2 - Epoch {epoch+1}/15, Loss: {total_loss/len(train_ds_focus_aug):.4f}")

Starting Focused Stage 1 (Focus Class: cat)...
Focused Stage 1 - Epoch 1/10, Loss: 1.3396
Focused Stage 1 - Epoch 2/10, Loss: 0.9350
Focused Stage 1 - Epoch 3/10, Loss: 0.8199
Focused Stage 1 - Epoch 4/10, Loss: 0.7671
Focused Stage 1 - Epoch 5/10, Loss: 0.7372
Focused Stage 1 - Epoch 6/10, Loss: 0.6814
Focused Stage 1 - Epoch 7/10, Loss: 0.6691
Focused Stage 1 - Epoch 8/10, Loss: 0.6357
Focused Stage 1 - Epoch 9/10, Loss: 0.6284
Focused Stage 1 - Epoch 10/10, Loss: 0.6208

Starting Focused Stage 2 (Fine-tuning)...
Focused Stage 2 - Epoch 1/15, Loss: 0.5388
Focused Stage 2 - Epoch 2/15, Loss: 0.4872
Focused Stage 2 - Epoch 3/15, Loss: 0.4340
Focused Stage 2 - Epoch 4/15, Loss: 0.3865
Focused Stage 2 - Epoch 5/15, Loss: 0.3623
Focused Stage 2 - Epoch 6/15, Loss: 0.3443
Focused Stage 2 - Epoch 7/15, Loss: 0.3050
Focused Stage 2 - Epoch 8/15, Loss: 0.2820
Focused Stage 2 - Epoch 9/15, Loss: 0.2793
Focused Stage 2 - Epoch 10/15, Loss: 0.2509
Focused Stage 2 - Epoch 11/15, Loss: 0.2371
Focu

In [11]:
accs_new = per_class_accuracy(focused_aug_line, test_ds, y_test)

print("=== focused_aug_line per-class accuracy ===")
print_table(accs_new)

focused_aug_worst_acc = accs_new[FOCUS]

print("\n=== WORST LABEL ACCURACY COMPARISON ===")
print(f"Baseline {labels[worst_label]} acc: {baseline_worst_acc:.4f}")
print(f"focused_aug_line {labels[worst_label]} acc: {focused_aug_worst_acc:.4f}")

delta_pp = (focused_aug_worst_acc - baseline_worst_acc) * 100
print(f"Delta (pp): {delta_pp:.2f}")

required_pp = 10
print(f"Required (pp) > {required_pp}")
print("PASS" if delta_pp > required_pp else "NOT PASS")

=== focused_aug_line per-class accuracy ===
 0 airplane     87.90%
 1 automobile   94.80%
 2 bird         84.10%
 3 cat          85.10%
 4 deer         80.40%
 5 dog          80.70%
 6 frog         91.60%
 7 horse        88.00%
 8 ship         95.80%
 9 truck        91.70%

=== WORST LABEL ACCURACY COMPARISON ===
Baseline cat acc: 0.6930
focused_aug_line cat acc: 0.8510
Delta (pp): 15.80
Required (pp) > 10
PASS


## 3. Infrastructure & Regularization Optimization

In this final stage, I enhanced the model's architecture and applied regularization techniques to further improve generalization performance, building upon the previous data-centric improvements.

### 1) Classification Head Strengthening (Infrastructure Optimization)
* **Architecture:** Replaced the single linear layer with a deeper, multi-layer MLP head:
    * **BatchNorm1d:** To stabilize the distribution of features from the backbone.
    * **Dropout (0.25):** To prevent overfitting by randomly deactivating neurons during training.
    * **Hidden Layer (512 units):** To allow for more complex feature mapping before the final output.

### 2) Label Smoothing (Regularization)
* **Implementation:** Applied `label_smoothing=0.02` within the `CrossEntropyLoss`.
* **The Reason:** Usually, the model tries to be completely certain about one answer. This can lead to overconfidence and fitting too closely to the training set. By making the target labels a bit softer, like changing them from 1.0 to 0.98, I encouraged the model to be more flexible. This helps the model perform better on new images and keeps the decision boundaries from becoming too sharp or rigid.

### 3) Training Consistency
* All previous optimizations—**Data Augmentation, Oversampling, and Class Weights (3.0x)**—were maintained to ensure a cumulative performance gain.

In [12]:
base3 = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

for param in base3.parameters():
    param.requires_grad = False

# diff_4: Add a stronger classification head (infrastructure optimization)
in_features = base3.fc.in_features
base3.fc = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.25),      
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.25),      
    nn.Linear(512, num_classes)
)

final_model = base3.to(device)

# diff_5: Apply label smoothing to reduce overconfidence and improve generalization
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.02)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, final_model.parameters()), lr=1e-3)

print(f"Starting Final Stage 1 (Head Strengthening)...")
for epoch in range(10):
    final_model.train()
    total_loss = 0
    for x, y in train_ds_focus_aug:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = final_model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Final Stage 1 - Epoch {epoch+1}/10, Loss: {total_loss/len(train_ds_focus_aug):.4f}")
        
# Unfreeze last 40 parameters
params = list(final_model.parameters())
for p in params[:-40]: p.requires_grad = False
for p in params[-40:]: p.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=2e-5
)

print(f"\nStarting Final Stage 2 (Deep Refinement with Label Smoothing)...")
for epoch in range(15):
    final_model.train()
    total_loss = 0
    for x, y in train_ds_focus_aug:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = final_model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Final Stage 2 - Epoch {epoch+1}/15, Loss: {total_loss/len(train_ds_focus_aug):.4f}")

Starting Final Stage 1 (Head Strengthening)...
Final Stage 1 - Epoch 1/10, Loss: 1.1286
Final Stage 1 - Epoch 2/10, Loss: 0.9245
Final Stage 1 - Epoch 3/10, Loss: 0.8595
Final Stage 1 - Epoch 4/10, Loss: 0.8526
Final Stage 1 - Epoch 5/10, Loss: 0.8212
Final Stage 1 - Epoch 6/10, Loss: 0.7966
Final Stage 1 - Epoch 7/10, Loss: 0.7806
Final Stage 1 - Epoch 8/10, Loss: 0.7628
Final Stage 1 - Epoch 9/10, Loss: 0.7485
Final Stage 1 - Epoch 10/10, Loss: 0.7417

Starting Final Stage 2 (Deep Refinement with Label Smoothing)...
Final Stage 2 - Epoch 1/15, Loss: 0.6817
Final Stage 2 - Epoch 2/15, Loss: 0.6333
Final Stage 2 - Epoch 3/15, Loss: 0.6034
Final Stage 2 - Epoch 4/15, Loss: 0.5903
Final Stage 2 - Epoch 5/15, Loss: 0.5591
Final Stage 2 - Epoch 6/15, Loss: 0.5318
Final Stage 2 - Epoch 7/15, Loss: 0.5208
Final Stage 2 - Epoch 8/15, Loss: 0.5152
Final Stage 2 - Epoch 9/15, Loss: 0.4932
Final Stage 2 - Epoch 10/15, Loss: 0.4699
Final Stage 2 - Epoch 11/15, Loss: 0.4658
Final Stage 2 - Epoch 1

In [13]:
accs_new2 = per_class_accuracy(final_model, test_ds, y_test)

print("=== final_model per-class accuracy ===")
print_table(accs_new2)

final_model_worst_acc = accs_new2[FOCUS]

print("\n=== WORST LABEL ACCURACY COMPARISON (FINAL) ===")
print(f"Baseline {labels[worst_label]} acc: {baseline_worst_acc:.4f}")
print(f"Final Model {labels[worst_label]} acc: {final_model_worst_acc:.4f}")

delta_pp_final = (final_model_worst_acc - baseline_worst_acc) * 100
print(f"Delta (pp): {delta_pp_final:.2f}")

required_pp = 10

print(f"Required (pp) > {required_pp}")
print("PASS" if delta_pp_final > required_pp else "NOT PASS")

=== final_model per-class accuracy ===
 0 airplane     87.00%
 1 automobile   93.00%
 2 bird         77.80%
 3 cat          86.10%
 4 deer         80.80%
 5 dog          74.20%
 6 frog         92.50%
 7 horse        85.90%
 8 ship         93.30%
 9 truck        94.00%

=== WORST LABEL ACCURACY COMPARISON (FINAL) ===
Baseline cat acc: 0.6930
Final Model cat acc: 0.8610
Delta (pp): 16.80
Required (pp) > 10
PASS


In [14]:
base_list = accs_base    
focused_list = accs_new  
final_list = accs_new2   

def print_model_summary(acc_dict, model_name):
    accuracies = list(acc_dict.values())
    mean_acc = sum(accuracies) / len(accuracies)
    
    print(f"[{model_name}]")
    print(f"- Mean Accuracy: {mean_acc * 100:.2f}%")
    print("-" * 30)


print_model_summary(base_list, "Baseline Model")
print_model_summary(focused_list, "Focused Aug Model")
print_model_summary(final_list, "Final Optimized Model")

base_mean = sum(base_list.values()) / len(base_list)
final_mean = sum(final_list.values()) / len(final_list)

print("=== IMPROVEMENT SUMMARY ===")
print(f"Overall Mean Accuracy Delta (Total): {(final_mean - base_mean) * 100:+.2f} pp")

[Baseline Model]
- Mean Accuracy: 83.32%
------------------------------
[Focused Aug Model]
- Mean Accuracy: 88.01%
------------------------------
[Final Optimized Model]
- Mean Accuracy: 86.46%
------------------------------
=== IMPROVEMENT SUMMARY ===
Overall Mean Accuracy Delta (Total): +3.14 pp


## 4.Experimental Results Analysis & Discussion

### 1) Performance Gains by Stage
* **Stage 2 (Focused Augmentation):**
    * **Results:** The mean accuracy improved from **83.32% to 88.01%**, and the **'Cat' class** accuracy significantly increased by **15.80 percentage points** (from 69.3% to 85.1%).
    * **Analysis:** This overall improvement came from a focus on data. Doubling the 'Cat' samples directly boosted its performance. Applying **Random Resized Crop** and **Color Jitter** to the **entire dataset improved the model's ability** to extract features. This resulted in better performance across all classes.



* **Stage 3 (Final Optimized Model):**
    * **Results:** The **'Cat' class** accuracy reached its peak at **86.1%** (+1pp compared to Stage 2). However, the **overall mean accuracy dropped** to **86.46%**.
    * **Analysis:** The introduction of the **MLP Head** increased the model's capacity and made it more sensitive to the **Class Weight (3.0x)** assigned to cats. As a result, the **model focused heavily on 'Cat' features** during optimization. In addition, the **Label Smoothing** effect likely made the model more cautious with ambiguous samples from other classes, leading to a slight drop in their respective accuracies.



### 2) Key Takeaways & Conclusion
This experiment demonstrated the powerful synergy between data-level optimization and model-level refinement.

1.  **Effectiveness of Data-Centric Strategies:** I learned that targeted **oversampling** is essential for balancing specific underperforming labels. Moreover, universal augmentations like **Color Jitter and Random Resized Crop** are critical for building a robust model that generalizes well across the entire dataset.
2.  **Infrastructure vs. Class Weight Interaction:** Strengthening the classification head (MLP) can amplify the effects of class weights. While this is highly effective for maximizing performance on a specific target, it can lead to a trade-off where the model "over-focuses" on the weighted class at the expense of others.
3.  **Final Evaluation:** If the sole objective is to maximize 'Cat' class accuracy, the **Final Model** is superior. However, considering the overall balance and system-wide reliability, the **Stage 2 (Focused Augmentation)** model is arguably the most suitable architecture for general deployment.